# Sophia AGI — LoRA training (Google Colab)

## ⚠️ Enable GPU **before** running code

1. Menu **Runtime** → **Change runtime type**
2. **Hardware accelerator:** `T4 GPU` (free tier) or `L4 GPU`
3. Click **Save**
4. **Runtime** → **Restart session**
5. Run cells from the top

Train **Sophia** adapter on the provenance corpus with QLoRA (4-bit).

**Repo:** [github.com/tomyimkc/sophia-agi](https://github.com/tomyimkc/sophia-agi)

In [ ]:
import torch

if not torch.cuda.is_available():
    from IPython.display import display, Markdown
    display(Markdown("""
### ❌ No GPU detected

You are on **CPU runtime**. Training will not work.

1. **Runtime** → **Change runtime type**
2. Hardware accelerator: **T4 GPU**
3. **Save** → **Runtime** → **Restart session**
4. Re-run this cell — you should see a GPU name (e.g. Tesla T4)
    """))
    raise SystemExit("Stop: enable GPU runtime, restart, re-run")

print("✅ GPU:", torch.cuda.get_device_name(0))
print("   VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
import os
import shutil
import subprocess

REPO = "sophia-agi"
REMOTE = "https://github.com/tomyimkc/sophia-agi.git"

if os.path.isdir(REPO):
    pull = subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], capture_output=True, text=True)
    if pull.returncode != 0:
        print("git pull failed — recloning")
        shutil.rmtree(REPO)
    else:
        print(pull.stdout.strip() or "sophia-agi already up to date")

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", REMOTE], check=True)

%cd sophia-agi
!git log -1 --oneline

In [ ]:
# Pin versions for Colab stability (T4 + QLoRA)
!pip install -q -U \
  "transformers>=4.44,<5.0" \
  "peft>=0.10,<0.18" \
  "trl>=0.11,<0.20" \
  "datasets>=2.18" \
  "accelerate>=0.28" \
  "bitsandbytes>=0.43"

import transformers, trl, peft, bitsandbytes
print("transformers", transformers.__version__)
print("trl", trl.__version__)
print("peft", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)

In [ ]:
import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
# T4 (~15 GB): stay on 3B. Use 7B only on L4/A100-class GPUs.
MODEL = "Qwen/Qwen2.5-7B-Instruct" if vram_gb > 20 else "Qwen/Qwen2.5-3B-Instruct"
BATCH_SIZE = 1
MAX_SEQ_LEN = 1024
print(f"Using {MODEL} ({vram_gb:.1f} GB VRAM)")
print(f"batch_size={BATCH_SIZE} max_seq_len={MAX_SEQ_LEN}")

In [ ]:
import subprocess
import sys
from pathlib import Path

prep = subprocess.run([sys.executable, "tools/prepare_lora_dataset.py"], check=False)
if prep.returncode != 0:
    raise RuntimeError(f"prepare_lora_dataset failed (exit {prep.returncode})")

cmd = [
    sys.executable,
    "tools/train_lora.py",
    "--4bit",
    "--epochs",
    "3",
    "--model",
    MODEL,
    "--batch-size",
    str(BATCH_SIZE),
    "--max-seq-len",
    str(MAX_SEQ_LEN),
]
print("Running:", " ".join(cmd))
train = subprocess.run(cmd, check=False)
if train.returncode != 0:
    raise RuntimeError(
        f"train_lora failed (exit {train.returncode}). Scroll up for traceback."
    )

ckpt = Path("training/lora/checkpoints/sophia-v1")
required = ["adapter_config.json", "adapter_model.safetensors"]
missing = [f for f in required if not (ckpt / f).exists()]
if missing:
    raise RuntimeError(f"Training incomplete — missing {missing}")
print("✅ Training complete:", sorted(p.name for p in ckpt.iterdir()))

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

ckpt = Path("training/lora/checkpoints/sophia-v1")
required = ["adapter_config.json", "adapter_model.safetensors"]
missing = [f for f in required if not (ckpt / f).exists()]
if missing:
    raise RuntimeError(
        f"Training incomplete — missing {missing}. "
        "Run the **train cell above** and wait for '✅ Training complete'."
    )
print("Checkpoint OK:", sorted(p.name for p in ckpt.glob("*")))

!python tools/claude_model_lab.py write-modelfile --adapter training/lora/checkpoints/sophia-v1
zip_path = Path("sophia-lora-v1.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive("sophia-lora-v1", "zip", str(ckpt))
zip_mb = round(zip_path.stat().st_size / 1e6, 2)
print("Zip size MB:", zip_mb)
if zip_mb < 1:
    raise RuntimeError("Zip too small — training did not produce adapter weights.")
files.download("sophia-lora-v1.zip")

## After download (local)

```bash
unzip sophia-lora-v1.zip -d training/lora/checkpoints/sophia-v1
python tools/eval_local_model.py --adapter training/lora/checkpoints/sophia-v1 --with-gate
ollama create sophia-7b -f models/ollama/Modelfile
```

Always run **sophia_gate_check** on production answers.